# FUNAAB ICTREC Helpdesk Predictive Modeling
This notebook demonstrates the end-to-end workflow and evaluates both the Baseline and Advanced models using Orange Toolkit metrics (AUC, CA, F1, Prec, Recall, MCC) and Confusion Matrices.

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix

warnings.filterwarnings('ignore')

# Load dataset
file_path = "FUNAAB_ICTREC_Helpdesk_500_Instances.csv"
df_raw = pd.read_csv(file_path)

def categorize_resolution(hrs):
    if hrs <= 8:
        return 'Fast (< 8 hrs)'
    else:
        return 'Medium (8 hrs - 1 week)'

df = df_raw.copy()
df['Resolution_Category'] = df['Resolution_Time_Hrs'].apply(categorize_resolution)

# Target: 1 for 'Medium', 0 for 'Fast'
y = np.where(df['Resolution_Category'] == 'Medium (8 hrs - 1 week)', 1, 0)
print("Data loaded and target categorized.")

## 1. Baseline Model (Random Forest)
Training the baseline model, saving predictions, and generating the confusion matrix.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

features = ['User_Type', 'Unit_Dept', 'Issue_Category', 'Priority']
X_base = pd.get_dummies(df[features], drop_first=True)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_base, y, test_size=0.2, random_state=42)

clf_base = RandomForestClassifier(n_estimators=100, random_state=42)
clf_base.fit(X_train_b, y_train_b)

y_pred_base = clf_base.predict(X_test_b)
y_prob_base = clf_base.predict_proba(X_test_b)[:, 1]

# Save outputs
results_base = df.iloc[X_test_b.index].copy()
results_base['Actual_Class'] = y_test_b
results_base['Predicted_Class'] = y_pred_base
results_base['Prob_Delayed'] = y_prob_base
results_base.to_csv("Baseline_Predictions.csv", index=False)
print("Baseline predictions saved to 'Baseline_Predictions.csv'")

# Confusion Matrix
cm_base = confusion_matrix(y_test_b, y_pred_base)
print("\n--- Baseline Confusion Matrix ---")
display(pd.DataFrame(cm_base, index=['Actual Fast (0)', 'Actual Medium (1)'], columns=['Pred Fast (0)', 'Pred Medium (1)']))

## 2. Advanced Model (Hybrid NLP + XGBoost)
Training the advanced model with cost-sensitive learning, saving predictions, and generating the confusion matrix.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb

text_feature = 'Issue_Description'
cat_features = ['User_Type', 'Unit_Dept', 'Issue_Category', 'Priority']
X_adv = df[[text_feature] + cat_features]

preprocessor = ColumnTransformer(transformers=[
    ('text', TfidfVectorizer(max_features=150, stop_words='english'), text_feature),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_adv, y, test_size=0.2, random_state=42)

clf_adv = xgb.XGBClassifier(n_estimators=150, random_state=42, scale_pos_weight=2.5, eval_metric='logloss', max_depth=4)
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf_adv)])
pipeline.fit(X_train_a, y_train_a)

y_pred_adv = pipeline.predict(X_test_a)
y_prob_adv = pipeline.predict_proba(X_test_a)[:, 1]

# Save outputs
results_adv = df.iloc[X_test_a.index].copy()
results_adv['Actual_Class'] = y_test_a
results_adv['Predicted_Class'] = y_pred_adv
results_adv['Prob_Delayed'] = y_prob_adv
results_adv.to_csv("Advanced_Predictions.csv", index=False)
print("Advanced predictions saved to 'Advanced_Predictions.csv'")

# Confusion Matrix
cm_adv = confusion_matrix(y_test_a, y_pred_adv)
print("\n--- Advanced Confusion Matrix ---")
display(pd.DataFrame(cm_adv, index=['Actual Fast (0)', 'Actual Medium (1)'], columns=['Pred Fast (0)', 'Pred Medium (1)']))

## 3. Orange Toolkit Format Evaluation
Comparing AUC, Classification Accuracy (CA), F1, Precision, Recall, and MCC.

In [ ]:
def evaluate_model(y_true, y_pred, y_prob):
    return {
        'AUC': roc_auc_score(y_true, y_prob),
        'CA': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'Prec': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred)
    }

results = {
    'Baseline (Random Forest)': evaluate_model(y_test_b, y_pred_base, y_prob_base),
    'Advanced (XGBoost + NLP)': evaluate_model(y_test_a, y_pred_adv, y_prob_adv)
}

eval_df = pd.DataFrame(results).T
print("\n--- Model Evaluation Comparison ---")
display(eval_df.round(3))